   Author: Ankit Kariryaa, University of Bremen

In [1]:
import rasterio                  # I/O raster data (netcdf, height, geotiff, ...)
import rasterio.warp             # Reproject raster samples
from rasterio import windows
import fiona                     # I/O vector data (shape, geojson, ...)
import pyproj                    # Change coordinate reference system
import geopandas
import shapely
from shapely.geometry import Point
import imgaug as ia
from imgaug import augmenters as iaa

import numpy as np               # numerical array manipulation
import os
import glob
from tqdm import tqdm
from PIL import Image
from itertools import product
from tensorflow.keras.models import load_model

from skimage.segmentation import mark_boundaries
from sklearn.model_selection import train_test_split


%matplotlib inline
import matplotlib.pyplot as plt  # plotting tools


import warnings                  # ignore annoying warnings
warnings.filterwarnings("ignore")
import logging
logger = logging.getLogger()
logger.setLevel(logging.CRITICAL)

%reload_ext autoreload
%autoreload 2
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

In [2]:
def get_tiles(ds, width=256, height=256):
    nols, nrows = ds.meta['width'], ds.meta['height']
    offsets = product(range(0, nols, width), range(0, nrows, height))
    big_window = windows.Window(col_off=0, row_off=0, width=nols, height=nrows)
    for col_off, row_off in  offsets:
        window =windows.Window(col_off=col_off, row_off=row_off, width=width, height=height).intersection(big_window)
        transform = windows.transform(window, ds.transform)
        yield window, transform


In [16]:

def split_image(base_dir, fn, out_path, out_fn, tile_width=5000, tile_height=5000):
    with rasterio.open(os.path.join(base_dir, fn)) as inds:
        meta = inds.meta.copy()
        for window, transform in get_tiles(inds, tile_width, tile_height ):
            print(window)
            meta['transform'] = transform
            meta['width'], meta['height'] = window.width, window.height
            outpath = os.path.join(out_path,out_fn.format(int(window.col_off), int(window.row_off)))
            with rasterio.open(outpath, 'w', **meta) as outds:
                outds.write(inds.read(window=window))

In [ ]:
# reproject and add NDVI
rootPath = '/media/xiaoye/ps/nicfi'
mosaicName = 'planet_medres_normalized_analytic_2019-06_2019-11_mosaic'
searchPath = f'{rootPath}/{mosaicName}/fix'
subfolders = [f.path for f in os.scandir(searchPath) if f.is_dir()]

for i in subfolders:
    base_dir = i
    nicfiMosaic = glob.glob(f'{i}/nicif*.tif',recursive = True)
    nicfi_fn = os.path.basename(nicfiMosaic[0])

    out_path = f'{base_dir}/cutouts'
    if not os.path.exists(out_path):
        os.makedirs(out_path)
        ndvi_output_fn = 'split_{}-{}.tif'
        split_image(base_dir, nicfi_fn, out_path, ndvi_output_fn,10000,10000)
    else:
        print ('File already splited')

## Compress files

In [43]:
# reproject and add NDVI
rootPath = '/media/xiaoye/ps/nicfi'
mosaicName = 'planet_medres_normalized_analytic_2019-06_2019-11_mosaic'
searchPath = f'{rootPath}/{mosaicName}/fix/sahel'
subfolders = [f.path for f in os.scandir(searchPath) if f.is_dir()]

for i in subfolders:
    base_dir = i
    nicfiMosaic = glob.glob(f'{i}/*merged.tif',recursive = True)
    processed = glob.glob(f'{i}/*compressed.tif',recursive = True)
    runCompress = True if nicfiMosaic != [] else False
    proExist = True if processed == [] else False
    print (proExist)
    if runCompress and proExist:
        outfn = f'{i}/crop_merged_compressed.tif'
        with rasterio.open(nicfiMosaic[0],'r+') as src:
            data = src.read()
            outMeta = src.meta.copy()
            outMeta.update({'tiled': True,
                'blockxsize': 256,
                'blockysize': 256,
                'compress': 'lzw',
                'nodata': 0})
            with rasterio.open(outfn,'w', **outMeta) as dest:
                dest.write(data)
    else:
        print (f'{i} not processed yet')


False
/media/xiaoye/ps/nicfi/planet_medres_normalized_analytic_2019-06_2019-11_mosaic/fix/sahel/00015_-0004 not processed yet
False
/media/xiaoye/ps/nicfi/planet_medres_normalized_analytic_2019-06_2019-11_mosaic/fix/sahel/00016_00011 not processed yet
False
/media/xiaoye/ps/nicfi/planet_medres_normalized_analytic_2019-06_2019-11_mosaic/fix/sahel/00014_-0017 not processed yet
False
/media/xiaoye/ps/nicfi/planet_medres_normalized_analytic_2019-06_2019-11_mosaic/fix/sahel/00016_00012 not processed yet
False
/media/xiaoye/ps/nicfi/planet_medres_normalized_analytic_2019-06_2019-11_mosaic/fix/sahel/00014_-0001 not processed yet
False
/media/xiaoye/ps/nicfi/planet_medres_normalized_analytic_2019-06_2019-11_mosaic/fix/sahel/00016_00023 not processed yet
False
/media/xiaoye/ps/nicfi/planet_medres_normalized_analytic_2019-06_2019-11_mosaic/fix/sahel/00015_00041 not processed yet
False
/media/xiaoye/ps/nicfi/planet_medres_normalized_analytic_2019-06_2019-11_mosaic/fix/sahel/00017_-0006 not proces